## Naivni Bajesov algoritam

Problem klasifikacije možemo da predstavimo kao određivanje uslovne verovatnoće ciljne promenljive $y$ pri uslovu atributa $X$ $p(y|X)$ (kao i obično, skraćeno pišemo $X$ umesto $X_1, X_2, ..., X_n$).

Po definicije uslovne verovatnoće
$$p(y|X) = \frac{p(X, y)}{p(X)}$$

Kako biste ocenili ovu verovatnoću za dati skup podataka?
Mozemo da pokušamo brojanjem - (koliko puta se zajedno javljaju odgovarajuće vrednosti $y$ i svih atributa $X_i$) podeljeno sa (koliko puta se zajedno javljaju odgovarajuće vrednosti svih atributa $X_i$). Ali što više atributa $X_i$ imamo, to će brojanje imati manje smisla, tj. nećemo imati dovoljno instanci koje pokrivaju sve moguće kombinacije da ocenimo verovatnoću ispravno. Moramo da pokušamo da olakšamo problem.

Simetrična formula za uslovnu verovatnoću $X$ pri uslovu $y$:
$$p(X|y) = \frac{p(X, y)}{p(y)}$$
Ako to zamenimo u $p(y|X)$
$$p(y|X) = \frac{p(X|y)p(y)}{p(X)}$$
Dobijamo Bajesovu formulu. Sada imamo tri verovatnoće koje treba da ocenimo da bismo dobili ono što nas zanima. Od te tri, jedna je laka $p(y)$ - pošto je jedna promenljiva u pitanju, možemo da ocenimo verovatnoću brojanjem. Ali, preostale dve verovatnoće su teške jer imamo puno atributa $X_i$. Stoga olakšavamo problem jednom _naivnom_ pretpostavkom - pretpostavljamo da su svi atributi $X_i$ uslovno nezavisni pri uslovu $y$, tj.
$$P(X|y) = \prod_{i=1..n}{P(X_i|y)}$$
Pojedinačnu uslovnu verovatnoću $P(X_i|y)$ opet lako možemo da ocenimo brojanjem jer je u pitanju samo jedan atribut $X_i$ i jedna ciljna promenljiva $y$.
Ako ovo zamenimo u Bajesovu formulu, dobijamo
$$p(y|X) = \frac{\prod_{i=1..n}{P(X_i|y)}p(y)}{p(X)}$$
Sada lako možemo da ocenimo čitav brojilac. Zašto ne možemo da razdvojimo $p(X) = \prod_{i=1..n}{P(X_i)}$? Zato što je nezavisnost jaci uslov od uslovne nezavisnosti. Dakle, imenilac ostaje težak. Međutim, kako je naš cilj da odredimo najverovatniju klasu, imenilac nam nije ni potreban jer ne zavisi od klase $y$.
$$p(y|X) \propto \prod_{i=1..n}{P(X_i|y)}p(y)$$
$$\hat{y} = argmax_{y}{p(y|X)} = argmax_{y}{\frac{\prod_{i=1..n}{P(X_i|y)}p(y)}{p(X)}} = argmax_{y}\prod_{i=1..n}{P(X_i|y)}p(y)$$


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

## Naivni Bajesov algoritam za kategoričke atribute

### Učitavanje i priprema podataka

Učitavamo podatke iz <code>ballons.csv</code>.

In [2]:
df = pd.read_csv('balloons.csv')

In [3]:
df.head()

,color,size,act,age,inflated
0,YELLOW,SMALL,STRETCH,ADULT,T
1,YELLOW,SMALL,STRETCH,ADULT,T
2,YELLOW,SMALL,STRETCH,CHILD,F
3,YELLOW,SMALL,DIP,ADULT,F
4,YELLOW,SMALL,DIP,CHILD,F


Osnovne informacije o atributima:

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   color     76 non-null     object
 1   size      76 non-null     object
 2   act       76 non-null     object
 3   age       76 non-null     object
 4   inflated  76 non-null     object
dtypes: object(5)
memory usage: 3.1+ KB


Koliko različitih vrednosti uzima svaki od atributa?

In [5]:
for feature_name in df.columns:
    print("{} - {}".format(feature_name, df[feature_name].unique()))

color - ['YELLOW' 'PURPLE']
size - ['SMALL' 'LARGE']
act - ['STRETCH' 'DIP']
age - ['ADULT' 'CHILD']
inflated - ['T' 'F']


Izdvajamo ulazne atribute i ciljni atribut (<code>'inflated'</code>).

In [6]:
X = df.drop('inflated', axis=1)
y = df['inflated']

In [7]:
X.shape

(76, 4)

Delimo skup podataka na trening i test skup.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=123)

In [9]:
X_train.shape

(53, 4)

In [10]:
X_test.shape

(23, 4)

Pre konstrukcije Naivnog Bajesovog modela potrebno je da pridružimo kategoričkim atributima numeričke reprezentacije.

In [11]:
from sklearn.preprocessing import OrdinalEncoder

In [12]:
encoder = OrdinalEncoder()

Kao i u slučaju drugih tehnika pretprocesiranja podataka, enkoder *fit*-ujemo samo na trening skupu.

In [13]:
encoder.fit(X_train)

,categories,'auto'
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


In [14]:
X_train = encoder.transform(X_train)
X_test = encoder.transform(X_test)

*NAPOMENA:* <code>OrdinalEncoder</code> se koristi za pridruživanje numeričkih vrednosti kategoričkim atributima koji imaju prirodno definisano uređenje (*ordinalni atributi*, atribute sa $k$ kategorija u brojeve $0, 1, ..., k-1$). U našem slučaju kategorički atributi nemaju uređenje, ali su svi binarni, tako da ga ipak možemo iskoristiti. Takođe, varijanta Naivnog Bajesovog modela za kategoričke atribute po konstrukciji ne pretpostavlja uređenje nad vrednostima atributa.

### Naivni Bajesov model - CategoricalNB

Svaka od različitih klasa iz <code>naive_bayes</code> modula na različit način modeluje $p(X_i|y)$. Sve ostalo im je zajedničko.

<code>CategoricalNB</code> koristi sledeću formulu:
$$p(X_i=t|y=c) = \frac{N_{tic} + \alpha}{N_c + \alpha n_i}$$
gde je $N_{tic}$ broj instanci klase $c$ kod kojih je vrednost atributa $X_i$ jednaka $t$, a
$N_c$ je broj instanci klase $c$. $\alpha$. iz formule je samo dodatak koji koristimo da ne bismo imali problem sa verovatnoćama $0$ i $1$.

Ovakva formula $$p(X_i=t|y=c) = \frac{N_{tic}}{N_c}$$
ima smisla, ali može lako da se desi da u trening skupu nemamo nijednu instancu neke klase $c$ kod koje je atribut $X_i$ jednak nekom $t$. Tada je $N_{tic} = 0$ i čitava verovatnoća $p(y|X)$ postaje $0$ zbog množenja nulom. To nam se ne sviđa jer zbog samo jednog atributa čitava verovatnoća postaje nula. Da bismo rešili ovaj problem koristimo _smoothing_, odnosno ostavljamo malu verovatnoću da se može desiti nešto što nemamo u trening skupu.
Npr. treba na osnovu trening skupa da izračunamo koja je verovatnoća da će sutra Sunce izaći. Svih $N$ instanci u trening skupu imaju istu vrednost - Sunce je izašlo. Dakle, verovatnoća je $1$. Ali to je previše isključivo, pa ako se pravimo da je verovatnoća izlaska Sunca $0.5$ svakog dana, možemo da uradimo sledeće:
$$p(izlazi\_sutra) = \frac{N+1}{N+2}$$
Dodajemo u brojilac $1$, to je opcija da Sunce izađe sutra, a u imenilac dodajemo $2$ pošto ukupno postoje dve opcije - da izađe i da ne izađe. U opštem slučaju:
$$\frac{N+1}{N+k}$$ gde je $k$ broj kategorija. Početna formula je upravo ovakva uz dodatak parametra $\alpha$.

In [15]:
from sklearn.naive_bayes import CategoricalNB

In [16]:
model_cnb = CategoricalNB()

In [17]:
model_cnb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None
,min_categories,None


Kako su sva četiri ulazna atributa i ciljni atribut binarni, u <code>category_count_</code> imamo četiri $2\times2$ matrice broja pojavljivanja $X_i$ i $y$, što nam suštinski omogućava da izračunamo verovatnoću $p(X_i|y)$.

In [18]:
model_cnb.category_count_

[array([[18., 11.],
        [ 8., 16.]]),
 array([[17., 12.],
        [ 6., 18.]]),
 array([[20.,  9.],
        [ 7., 17.]]),
 array([[11., 18.],
        [15.,  9.]])]

Članska promenljiva <code>class_count_</code> predstavlja broj instanci po klasi, tj. na osnovu čega se izračunava verovatnoća $p(y)$.

In [19]:
model_cnb.class_count_

array([29., 24.])

### Evaluacija modela

Na osnovu vrednosti iz <code>category_count_</code> i <code>class_count_</code> prilikom poziva <code>predict</code> metoda izračunava se verovatnoću $p(y|X)$ i dobija predikcija klase.

In [20]:
y_train_pred = model_cnb.predict(X_train)
y_test_pred = model_cnb.predict(X_test)

Matricu konfuzije na trening i test skupu:

In [21]:
confusion_matrix(y_train, y_train_pred)

array([[25,  4],
       [ 9, 15]])

In [22]:
confusion_matrix(y_test, y_test_pred)

array([[9, 3],
       [3, 8]])

## Naivni Bajesov algoritam za klasifikaciju teksta

### Učitavanje i priprema podataka

Skup podataka koji ćemo koristiti je podskup [Ebart](http://www.arhiv.rs) korpusa novinskih članaka na srpskom jeziku. Podskup ovog korpusa, Ebart-5, obuhvata pet različitih tematskih celina (rubrika) - sport, ekonomija, politika, hronika (i kriminal) i kultura (i zabava). Skup podataka sadrži 5235 novinskih članaka i podeljen je na podatke za treniranje i podatke za testiranje u odnosu 2:1.

Novinski članci su podeljeni prema klasi kojoj pripadaju u odgovarajuće poddirektorijume: <code>Ekonomija</code>, <code>HronikaKriminal</code>, <code>KulturaZabava</code>, <code>Politika</code> i <code>Sport</code>. Podaci su prethodno pretprocesirani - uklonjene su *stop* reči i svaka reč je zamenjena svojim gramatičkim korenom, a zatim je izvršeno prebrojavanje reči. Svakom članku odgovara zasebna *txt* datoteka gde u pojedinačnim redovima imamo informacije o svim rečima (tj. korenima reči) koje se pojavljuju u članku zajedno sa njihovim brojem pojavljivanja (frekvencijom).

In [23]:
import os

In [24]:
os.listdir('Ebart/VektoriEbart-5/Trening/')           #izlistava direktorijum na zadatoj putanji (~ls komanda)

['Politika', 'HronikaKriminal', 'Ekonomija', 'KulturaZabava', 'Sport']

In [25]:
def read_data(root_dir_path):   #root_dir - koreni direktorijum odgovarajuceg skupa podataka
    corpus = []         #ulazni atributi (X)
    classes = []        #ciljni atribut (Y)
    
    for class_name in os.listdir(root_dir_path):
        class_dir_path = os.path.join(root_dir_path, class_name)
        
        for file_name in os.listdir(class_dir_path):
            file_path = os.path.join(class_dir_path, file_name)
            word_counts = {}
            
            with open(file_path, 'r') as f:         #with blok se brine o resursima (ne moramo da zatvaramo fajl)
                for line in f:
                    word, count = line.split()      #raspakivanje povratne vrednosti 
                    word_counts[word] = int(count)
            
            corpus.append(word_counts)
            classes.append(class_name)
            
    return corpus, classes

In [26]:
X_train, y_train = read_data('Ebart/VektoriEbart-5/Trening/')

In [27]:
len(X_train)

3492

In [28]:
X_test, y_test = read_data('Ebart/VektoriEbart-5/Testing/')

In [29]:
len(X_test)

1743

### Vektorizacija tekstualnih podataka

Hoćemo da napravimo _TF_ matricu (term matricu) kod koje su kolone reči, redovi dokumenti, a vrednost polja $(i,j)$ je broj pojavljivanja reči $j$ u dokumentu $i$. Pokušajte da transformišete ovu matricu u _TF-IDF_ matricu i proverite kako to utiče na rezultate klasifikacije.

S obzirom da su učitani podaci u vidu liste rečnika oblika reč-frekvencija, koristimo <code>DictVectorizer</code> za dobijanje vektorskih reprezentacija. <code>DictVectorizer</code> transformiše listu mapiranja atribut-vrednost (kod nas reč-frekvencija reči) u vektor vrednosti.

<img src="assets/DictVectorizer.png" width=850 align=left>

In [30]:
from sklearn.feature_extraction import DictVectorizer        

In [31]:
vectorizer = DictVectorizer()        

In [32]:
vectorizer.fit(X_train)

,dtype,<class 'numpy.float64'>
,separator,'='
,sparse,True
,sort,True


Članska promenljiva <code>feature_names_</code> sadrži listu izdvojenih atributa (reči).

In [33]:
vectorizer.feature_names_

['ab',
 'abasu',
 'abati',
 'abc',
 'abdul',
 'abdulah',
 'abe',
 'aberdin',
 'abhaziji',
 'abida',
 'aboliranxa',
 'abom',
 'abonmana',
 'abortus',
 'abramovicyu',
 'abrasxevicx',
 'abs',
 'abu',
 'ac',
 'aca',
 'acb',
 'acdi',
 'ace',
 'acetylene',
 'ackovicx',
 'acom',
 'acovicx',
 'acta',
 'acu',
 'acxima',
 'acximov',
 'acximovicx',
 'acxina',
 'acye',
 'acyucene',
 'ad',
 'ada',
 'adaldxize',
 'adamicya',
 'adamovicx',
 'adams',
 'adamson',
 'adana',
 'adancyicx',
 'adaptaciju',
 'adaptiranu',
 'adaptivnost',
 'addicted',
 'adekvanti',
 'adekvatan',
 'adekvatnijeg',
 'adekvatnom',
 'adele',
 'adelman',
 'adem',
 'adenauer',
 'adere',
 'adica',
 'adidas',
 'adidye',
 'adijano',
 'adili',
 'adilija',
 'adios',
 'adis',
 'aditivi',
 'administracija',
 'administrativnih',
 'administrator',
 'admiral',
 'adnan',
 'adolescent',
 'adolfa',
 'adore',
 'adorno',
 'adrana',
 'adrenalin',
 'adresa',
 'adresom',
 'adria',
 'adriakoop',
 'adrijano',
 'ads',
 'adut',
 'adventiskicyke',
 'adven

Broj atributa tj. broj različitih reči u celom skupu podataka (veličina vokabulara):

In [34]:
len(vectorizer.feature_names_)

36830

Kreiranim enkoderom transformišemo i trening i test skup.

In [35]:
X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

Kako se u većini dokumenata pojavljuje samo mali deo skupa svih mogućih reči, matrica vektorskih reprezentacija obično sadrži veliki broj nula i za takve matrice kažemo da su **retke**/**proredjene** (eng. *sparse*). Retke matrice se obično čuvaju u nekom kompresovanom formatu koji podrazumeva čuvanje samo trojki (row_index, column_index, value) za ne-nula vrednosti matrice. Metod <code>transform()</code> vraća matricu vektorskih reprezentacija u **CSR formatu** (eng. *Compressed Sparse Row format*). 

In [36]:
X_train              #obratiti pažnju na dimenzije rezultujuće retke matrice (broj_dokumenata, veličina_vokabulara)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 316178 stored elements and shape (3492, 36830)>

In [37]:
3492*36830

128610360

Metodima <code>toarray()</code> i <code>todense()</code> retke matrice se mogu razviti iz kompresovanog formata u pun oblik (gustu reprezentaciju). Metod <code>toarray()</code> vraca <code>numpy</code> niz, dok metod <code>todense()</code> vraca <code>sklearn</code> matricu.

In [38]:
X_train.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

Zarad lakse manipulacije podacima, možemo preći na <code>DataFrame</code> strukture podataka.

In [39]:
X_train = pd.DataFrame(X_train.toarray(), columns=vectorizer.feature_names_)
X_test = pd.DataFrame(X_test.toarray(), columns=vectorizer.feature_names_)

In [40]:
X_train.head()

,ab,abasu,abati,abc,abdul,abdulah,abe,aberdin,abhaziji,abida,...,zxurno,zxustel,zxustrine,zxustro,zxuticx,zxutih,zxutilovine,zxuto,zxutra,zxuzxa
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Naivni Bajesov model - MultinomialNB

<code>MultinomialNB</code> koristi sledeću formulu za računanje uslovne verovatnoće:
$$p(X_i=t|y=c) = \frac{N_{ci}+\alpha}{N_c + \alpha n}$$
gde je $N_{ci} = \sum_{x \in X_{train}}{x_i}$, dakle, zbir vrednosti atributa $X_i$ u svim instancama klase $c$. U našem slučaju to je broj pojavljivanja reči $t$ u svim dokumentima koji su klase $c$. Npr. koliko puta se javlja reč gol u svim dokumentima o sportu. $N_c = \sum_{i=1}^{n}{N_{ci}}$ je ukupan zbir vrednosti svih atributa za klasu $c$. $\alpha$ je ponovo samo _smoothing_ parametar.

In [41]:
from sklearn.naive_bayes import MultinomialNB

In [42]:
model_mnb = MultinomialNB()

In [43]:
model_mnb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Broj pojavljivanja svake od klasa:

In [44]:
model_mnb.class_count_

array([333., 620., 627., 935., 977.])

Broj pojavljivanja svake od reči u svakoj od klasa:

In [45]:
model_mnb.feature_count_

array([[ 1.,  0.,  0., ...,  2.,  0.,  0.],
       [ 0.,  0.,  0., ...,  1.,  0.,  0.],
       [ 0.,  1.,  0., ...,  5.,  1.,  0.],
       [ 0.,  0.,  0., ...,  2.,  0.,  0.],
       [ 2.,  0.,  1., ..., 27.,  0.,  2.]])

### Evaluacija modela

Tačnost klasifikacije na trening i na test skupu:

In [46]:
model_mnb.score(X_train, y_train)

0.9401489117983963

In [47]:
model_mnb.score(X_test, y_test)

0.8995983935742972

Matrica konfuzije na trening i na test skupu:

In [48]:
y_train_pred = model_mnb.predict(X_train)
y_test_pred = model_mnb.predict(X_test)

In [49]:
confusion_matrix(y_train, y_train_pred)

array([[312,   4,   3,  14,   0],
       [ 13, 504,   5,  97,   1],
       [  1,   4, 614,   6,   2],
       [  9,  28,   6, 885,   7],
       [  1,   4,   1,   3, 968]])

In [50]:
confusion_matrix(y_test, y_test_pred)

array([[152,   0,   1,  13,   0],
       [ 10, 226,   6,  66,   1],
       [  2,   0, 301,   6,   4],
       [  8,  36,   9, 411,   3],
       [  2,   1,   1,   6, 478]])

Da li su greške koje naš model pravi logične? 

In [51]:
model_mnb.classes_

array(['Ekonomija', 'HronikaKriminal', 'KulturaZabava', 'Politika',
       'Sport'], dtype='<U15')

Imamo značajan broj pogrešno klasifikovanih članaka iz klase HronikaKriminal u klasu Politika i obrnuto. Takođe imamo i nezanemarljiv broj pogrešno klasifikovanih članaka klase Ekonomija u klasu Politika. Ali to su vrlo često povezane teme, tako da može se reći da su greške koje model pravi opravdane (logične).

### Poređenje sa drugim modelima (KNN i stablo odlučivanja)

S obzirom na to da su sada <code>X_train</code> i <code>X_test</code> "obične" matrice, možemo da klasifikujemo ove podatke i nekim drugim modelima i da uporedimo rezultate.

In [52]:
from sklearn.neighbors import KNeighborsClassifier

In [53]:
model_knn = KNeighborsClassifier()

In [54]:
model_knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [55]:
y_pred = model_knn.predict(X_test)

Zašto _predict_ traje duže nego fit?

In [56]:
confusion_matrix(y_test, y_pred)

array([[ 46,   3,  15, 101,   1],
       [  3,  33,  31, 233,   9],
       [  0,   1, 172, 131,   9],
       [  4,  11,  17, 430,   5],
       [  1,   0,  13, 201, 273]])

In [57]:
accuracy_score(y_test, y_pred)

0.5473321858864028

In [58]:
from sklearn.tree import DecisionTreeClassifier

In [59]:
model_dt = DecisionTreeClassifier()

In [60]:
model_dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [61]:
y_pred = model_dt.predict(X_test)

In [62]:
confusion_matrix(y_test, y_pred)

array([[101,  15,   9,  29,  12],
       [ 15, 181,  12,  81,  20],
       [  6,   7, 256,  19,  25],
       [ 22,  67,  14, 335,  29],
       [  6,  12,   5,  15, 450]])

In [63]:
accuracy_score(y_test, y_pred)

0.7590361445783133

<div class='alert alert-info'>
DOMACI:
<ul>
    <li>Pokušajte da poboljšate ove rezultate optimizacijom hiperparametara.</li>
    <li>Isprobati <code>CountVectorizer</code> za vektorizaciju sirovih tekstualnih podataka.</li>
</ul>    
</div>

## Naivni Bajesov klasifikator za neprekidne atribute

### Učitavanje i priprema podataka

In [64]:
from sklearn.datasets import load_iris

In [65]:
data = load_iris()

In [66]:
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target, columns=['Species'])

In [67]:
X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=7)

### Naivni Bajesov model - GaussianNB

<code>GaussianNB</code> koristi sledeću formulu za računanje uslovne verovatnoće:
$$
P (X_i = x_i | y = c) = \frac{1}{\sqrt{2 \pi \sigma^2}} e^{ - \frac{(x_i - \mu_c)^2}{2\sigma_c^2}  }
$$
gde je $\mu_c$ prosečna vrednost atributa $X_i$ na instancama klase $c$, a $\sigma_c$ standardna devijacija atributa $X_i$ na instancama klase $c$.

In [69]:
from sklearn.naive_bayes import GaussianNB

In [70]:
model_gnb = GaussianNB()

In [71]:
model_gnb.fit(X_train, y_train.values.ravel())

,priors,None
,var_smoothing,1e-09


### Evaluacija modela

In [72]:
model_gnb.score(X_test, y_test)

0.9777777777777777

In [73]:
y_test_pred = model_gnb.predict(X_test)

In [74]:
confusion_matrix(y_test, y_test_pred)

array([[15,  0,  0],
       [ 0, 15,  0],
       [ 0,  1, 14]])